# Classificador de Espécies Iris — Google Colab

Notebook baseado no projeto `classifier` enviado. O fluxo reproduz as etapas implementadas no projeto:

1. **HU01 — Preparação:** carregar, validar e dividir o dataset Iris.
2. **HU02 — Treinamento:** comparar KNN, Árvore de Decisão, Naive Bayes e SVM com `GridSearchCV` e validação cruzada estratificada.
3. **HU03 — Avaliação:** calcular métricas, relatórios e matrizes de confusão.
4. **Exportação:** selecionar o melhor modelo pela acurácia média da validação cruzada e salvar `modelo_iris.pkl`.

O dataset Iris é carregado diretamente pelo scikit-learn, portanto não é necessário enviar um CSV.

## 1. Instalação e importações

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
os.makedirs("artifacts", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)
os.makedirs("reports/results", exist_ok=True)


## 2. HU01 — Carregamento e preparação dos dados

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

print(f"Quantidade de registros: {len(X)}")
print(f"Quantidade de atributos: {X.shape[1]}")
print("\\nDistribuição das classes:")
print(y.value_counts().sort_index().rename(index=dict(enumerate(iris.target_names))))

display(X.head())


### 2.1 Validação do dataset

In [ ]:
def validate_dataset(X, y):
    if X.empty or y.empty:
        raise ValueError("Dataset vazio.")
    if X.isnull().values.any():
        raise ValueError("Existem valores ausentes.")
    if not X.map(lambda value: pd.notna(value)).all().all():
        raise ValueError("Existem valores inconsistentes.")
    if not X.dtypes.apply(pd.api.types.is_numeric_dtype).all():
        raise ValueError("Atributos devem ser numéricos.")
    if not set(y.unique()).issubset({0, 1, 2}):
        raise ValueError("Espécies inválidas.")

validate_dataset(X, y)
print("Dataset validado com sucesso.")


### 2.2 Separação entre treino e teste
A divisão mantém 20% dos dados para teste e utiliza `stratify=y` para preservar a proporção das espécies.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)


## 3. HU02 — Definição dos modelos e hiperparâmetros

In [ ]:
models = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Decision Tree": Pipeline([
        ("scaler", "passthrough"),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", "passthrough"),
        ("model", GaussianNB())
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}

param_grids = {
    "KNN": {
        "scaler": [StandardScaler(), MinMaxScaler()],
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [None, 2, 3, 4, 5],
        "model__min_samples_split": [2, 4]
    },
    "Naive Bayes": {
        "model__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
    },
    "SVM": {
        "scaler": [StandardScaler(), MinMaxScaler()],
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

print("Modelos configurados:", ", ".join(models.keys()))


### 3.1 Treinamento com validação cruzada
Cada algoritmo é otimizado separadamente com `GridSearchCV`, utilizando 5 partições estratificadas e acurácia como métrica.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
cv_results = {}

for name, pipeline in models.items():
    print(f"Treinando {name}...")
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_
    cv_results[name] = {
        "accuracy": grid.best_score_,
        "parameters": grid.best_params_
    }

cv_summary = pd.DataFrame([
    {
        "model": name,
        "cv_accuracy": info["accuracy"],
        "best_parameters": info["parameters"]
    }
    for name, info in cv_results.items()
]).sort_values("cv_accuracy", ascending=False)

display(cv_summary)


## 4. HU03 — Avaliação dos modelos no conjunto de teste

In [ ]:
evaluation_results = []

for name, model in best_models.items():
    predictions = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, average="macro", zero_division=0),
        "recall": recall_score(y_test, predictions, average="macro", zero_division=0),
        "f1_score": f1_score(y_test, predictions, average="macro", zero_division=0)
    }
    evaluation_results.append(metrics)

    print(f"\\nRelatório de classificação — {name}")
    print(classification_report(
        y_test, predictions,
        target_names=iris.target_names,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, predictions, labels=[0, 1, 2])
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names
    )
    plt.title(f"Matriz de Confusão - {name}")
    plt.xlabel("Previsto")
    plt.ylabel("Real")
    plt.tight_layout()

    filename = f"reports/figures/{name.lower().replace(' ', '_')}.png"
    plt.savefig(filename, dpi=120)
    plt.show()
    plt.close()

results = pd.DataFrame(evaluation_results).sort_values("accuracy", ascending=False)
display(results)

results.to_csv("reports/results/resultados_modelos.csv", index=False)
print("Resultados salvos em reports/results/resultados_modelos.csv")


## 5. Seleção e exportação do melhor modelo
A seleção segue o projeto original: o melhor modelo é escolhido pela maior acurácia média obtida na validação cruzada, e não pela métrica do conjunto de teste.

In [ ]:
best_name = max(cv_results, key=lambda name: cv_results[name]["accuracy"])
best_model = best_models[best_name]

model_path = "artifacts/modelo_iris.pkl"
joblib.dump(best_model, model_path)

print(f"Melhor modelo: {best_name}")
print(f"Acurácia média na validação cruzada: {cv_results[best_name]['accuracy']:.4f}")
print(f"Modelo exportado em: {model_path}")


## 6. Teste de predição
A função abaixo mantém as validações e o formato de retorno do módulo `predictor.py` do projeto.

In [ ]:
FEATURE_NAMES = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
]

SPECIES = ["Iris Setosa", "Iris Versicolor", "Iris Virginica"]

def predict_species(model, features):
    if len(features) != 4:
        raise ValueError("São necessárias exatamente quatro características.")

    if any(
        not isinstance(value, (int, float, np.number)) or not pd.notna(value)
        for value in features
    ):
        raise ValueError("As características devem ser valores numéricos válidos.")

    if any(value <= 0 for value in features):
        raise ValueError("As características devem ser maiores que zero.")

    data = pd.DataFrame([features], columns=FEATURE_NAMES)
    prediction = model.predict(data)[0]
    probabilities = model.predict_proba(data)[0]

    return {
        "species": SPECIES[int(prediction)],
        "probabilities": {
            SPECIES[int(class_id)]: float(probability)
            for class_id, probability in zip(model.classes_, probabilities)
        }
    }

# Exemplo: medidas em centímetros
example = [5.1, 3.5, 1.4, 0.2]
prediction = predict_species(best_model, example)
prediction


## 7. Download dos artefatos
O arquivo `.pkl` pode ser utilizado posteriormente pelo backend Flask. O CSV contém as métricas dos modelos.

In [ ]:
from google.colab import files

files.download("artifacts/modelo_iris.pkl")


### Observações para integração com o backend

- O artefato exportado é um `Pipeline` completo, incluindo eventual etapa de normalização e o classificador.
- No backend, carregue o arquivo com `joblib.load()` e envie um DataFrame com as quatro colunas na mesma ordem e com os mesmos nomes usados no treinamento.
- Mantenha versões compatíveis de `scikit-learn` entre o ambiente de treinamento e o backend, pois arquivos serializados podem não ser compatíveis entre versões diferentes.
- Se quiser manter o artefato no Google Drive, monte o Drive no Colab e altere `model_path` para um caminho dentro de `/content/drive/MyDrive/`.